<a href="https://colab.research.google.com/github/kuroshkarimi/Machine-Learning-and-Data-Analysis/blob/main/Deep_Learning/Predicting_income_category.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting income category

The task is to develop a binary classification model that predicts whether a person’s annual income exceeds 50,000 dollars, using the Adult dataset, downloaded through scikit-learn’s OpenML interface. This project focuses on preparing numerical and categorical features, training a neural network, and comparing its performance with logistic regression.
Each row represents one person. Input features include the followings:

| Column | Meaning |
|---|---|
| **status** | **marital-status**: married, divorced, never married, widowed, etc. |
| **occupation** | Job category, such as sales, technical support, or managerial work. |
| **relationship** | Household or family role, such as husband, wife, own child, or other relative. |
| **race** | Recorded racial category. |
| **sex** | Recorded sex; the original dataset uses Female and Male. |
| **capital-gain** | Reported monetary gains from selling capital assets, such as investments. |
| **capital-loss** | Reported monetary losses from selling capital assets. |
| **hours-per-week** | Number of hours worked per week. |
| **native-country** | Country of origin, generally interpreted as country of birth. |
| **age** | Person’s age in years. |
| **workclass** | Employment category, such as private-sector, government, or self-employed. |
| **fnlwgt** | “Final weight”: a survey weight indicating how many people in the population the record represents. It is not income. |
| **education** | Highest recorded education level, such as high-school graduate, bachelor’s, or master’s. |
| **education-num** | An ordered numerical code for education level. It should not automatically be interpreted as years spent in education. |

The target is:
- 0: Annual income is at or below 50,000 dollars.
- 1: Annual income is above $50,000.

For each person, the model should produce an estimated probability of belonging to class 1. A chosen threshold then converts that score into a predicted income category.
Your implementation should accomplish the following:
1. Explore the dataset: examine feature types, missing values, class proportions, and relationships between features and income category.
2. Split the data: allocate 60% for training, 20% for validation, and 20% for testing, preserving approximately the same class proportions.
3. Prepare the inputs: impute missing numerical values, standardize numerical features, and one-hot encode categorical features. Fit preprocessing exclusively on the training set.
4. Establish a baseline: train logistic regression using the prepared features.
5. Train neural networks: compare the three proposed configurations—one hidden layer with 32 neurons, two layers with 64 and 32 neurons, and the same two-layer model with dropout.
6. Select the model: compare validation average precision, validation loss, and training time. Select a decision threshold on validation data if a specific precision–recall trade-off is required.
7. Evaluate the final model: report test average precision, ROC AUC, precision, recall, F1, and a confusion matrix.
8. Demonstrate prediction: pass a raw observation through the fitted preprocessing steps and obtain its predicted probability and category.
The main research question is:
Does a dense neural network improve income classification enough to justify its additional complexity compared with logistic regression?

A successful project does not require the neural network to win. It requires a fair comparison, preprocessing without data leakage, and a clear explanation of the results.
The final deliverable should be a notebook containing the analysis, preprocessing, training, evaluation, and conclusions. Because the dataset is historical, interpret the results as performance on this dataset—not evidence that the model reliably predicts present-day incomes.

In [1]:
# Importing required libraries

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
# Loading the data

adult = fetch_openml(
    name='adult',
    version=2,
    as_frame=True,

)
x = adult.data.copy()
target = adult.target.copy()

/usr/local/lib/python3.13/dist-packages/sklearn/datasets/_openml.py:110: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/list/data_name/adult/limit/2/data_version/2. Retrying...
  warn(


URLError: <urlopen error [Errno 110] Connection timed out>

In [ ]:
print(x.head(2))
print("\n")
print(target.head(2))

In [ ]:
# Checking some general properties of the data including the number of NaNs for each column

x.info()

In [ ]:
# Statistical properties of the data

x.describe(include = 'all')

In [ ]:
# Checking the target column

target.info()

In [ ]:
# Evaluating the population and percentage of each label category

print(target.value_counts())
target.value_counts(normalize = True)

In [ ]:
target

In [ ]:
# Converting the target column format to numbers

y = (target == '>50K').astype('int32')
np.unique(y, return_counts = True)

In [ ]:
# fnlwgt is a survey weighting variable.
# We omit it from this introductory, unweighted experiment.
x = x.drop(columns=["fnlwgt"])

In [ ]:
# Distinguishing the categorical and numerical columns

numeric_columns = x.select_dtypes(include = 'number').columns.tolist()
categorical_columns = x.select_dtypes(exclude = 'number').columns.tolist()
print(numeric_columns)
print(categorical_columns)

In [ ]:
# Taking a look at the unique elements of each column

for col in categorical_columns:
  print(f"{col} : {x[col].unique()}")

In [ ]:
# Splitting the data into training, validation and test sets as explained in the task

SEED = 42
x_train_val, x_test, y_train_val, y_test = train_test_split(x, y, test_size = 0.2, stratify = y, random_state = SEED)

x_train, x_val, y_train, y_val = train_test_split(x_train_val, y_train_val, test_size = 0.25, stratify = y_train_val, random_state = SEED)

print('shape of each section: \n', x_train.shape, x_val.shape, x_test.shape)
print('proportion of the training positives: ', np.mean(y_train))
print('proportion of the validation positives: ', np.mean(y_val))
print('proportion of the test positives: ', np.mean(y_test))

In [ ]:
# Creating pipelines for handling numeric and categorical columns easily

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([  ('imputer', SimpleImputer(strategy = 'median')),
                             ('standardization', StandardScaler())  ])



categorical_pipeline = Pipeline([ ('imputer', SimpleImputer(strategy = 'constant', fill_value = 'Missing')),
                                 ('onehot', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False)) ])

# sparse_output=False produces a conventional dense array for this small example.
# handle_unknown="ignore" allows an unseen category to be transformed without raising an error. Its one-hot
# strategy = 'constant' and fill_value = 'Missing' consider the constant character (Missing) instead of the nan values

In [ ]:
# Doing the transformation of the columns with respect to what we set in the Pipelines

from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer([  ('numeric', numeric_pipeline, numeric_columns),
                                  ('categorical', categorical_pipeline, categorical_columns)  ])

x_train_prepared = preprocessing.fit_transform(x_train)
x_val_prepared = preprocessing.transform(x_val)
x_test_prepared = preprocessing.transform(x_test)


In [ ]:
# Defining a baseline logistic regression algorithm, fitting the model and getting the probabilities
# on the validation data

from sklearn.linear_model import LogisticRegression
baseline = LogisticRegression(max_iter = 200, random_state = SEED)
baseline.fit(x_train_prepared, y_train)
baseline_prediction = baseline.pred_proba(x_val_prepared)[:, 1]
baseline_prediction
